In [ ]:
import ipywidgets as widgets
from types import ModuleType

isinstance(widgets, ModuleType)


In [ ]:
# Convert chess games

from humblemeister.data import ChessGameBank

cgb = ChessGameBank()
cgb.convert_games('env/data/chess_pgn_bank', 'env/data/chess_games', n_workers=24)


In [ ]:
# Generate shards of games in plain text PGN files

import os
import py7zr
import tempfile
import zipfile

file_index = 0
game_count = 0

ELO_THRESHOLD = 2200

def split_to_pgn(src_dir: str, dest_dir: str) -> None:
    global file_index
    global game_count

    # Read everything from 7z
    content = []
    for dir_path, _, files in os.walk(src_dir):
        for file in files:
            file_path = os.path.join(dir_path, file)
            print(file_path)
            
            with py7zr.SevenZipFile(file_path, "r") as zf:
                names = zf.namelist()
                if len(names) != 1:
                    return []
                with tempfile.TemporaryDirectory() as tmp:
                    zf.extractall(path=tmp)
                    with open(os.path.join(tmp, names[0]), "r", encoding="utf-8", errors="ignore") as f:
                        content = f.read().splitlines()
        
            os.makedirs(dest_dir, exist_ok=True)
        
            file_content = []
            acc = []
            black_elo = 0
            white_elo = 0
            for line in content:
                if line.startswith('[Event "') and len(acc) > 5:
                    if white_elo >= ELO_THRESHOLD and black_elo >= ELO_THRESHOLD:
                        file_content += acc
                        game_count += 1
                        if game_count >= 100000:
                            game_count = 0
                            with open(os.path.join(dest_dir, f'chess_games_{file_index:05}.pgn'), 'wt') as f:
                                for line in file_content:
                                    f.write(line)
                                    f.write('\n')
                            file_content = []
                            file_index += 1
                    white_elo = 0
                    black_elo = 0
                    acc = []
                elif line.startswith('[WhiteElo "'):
                    try:
                        white_elo = int(line[11 : line.index('"', 11)])
                    except:
                        while_elo = 0
                elif line.startswith('[BlackElo "'):
                    try:
                        black_elo = int(line[11 : line.index('"', 11)])
                    except (ValueError, IndexError):
                        black_elo = 0

                acc.append(line)
        
            if len(file_content) > 0:
                with open(os.path.join(dest_dir, f'chess_games_{file_index:05}.pgn'), 'wt') as f:
                    for line in file_content:
                        f.write(line)
                        f.write('\n')
            file_index += 1
    


In [ ]:
split_to_pgn(
    '/home/dev/dev_root/source/HumbleMeister/env/data/temp',
    '/home/dev/dev_root/source/HumbleMeister/env/data/chess_pgn_bank')